<a href="https://colab.research.google.com/github/albertocj1/Early_Warning_System_Dengue/blob/main/Copy_of_Dengue_Model_Final_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('/content/dengue-dataset-with-alert-epidemic.csv')
display(df.head())

,CITY,YEAR_WEEK,CASES,DEATHS,RAINFALL,TMAX,TMIN,TMEAN,RH,SUNSHINE,...,TMEAN_roll2_sum,TMEAN_roll4_sum,RH_roll2_mean,RH_roll4_mean,RH_roll2_sum,RH_roll4_sum,INCIDENCE_per_100k,RISK_LEVEL,ALERT,EPIDEMIC
0,CALOOCAN CITY,2016-W02,27,0,0.0,32.0,21.8,26.90,73.0,6.4,...,NaN,NaN,NaN,NaN,NaN,NaN,1.690776,Low,False,False
1,CALOOCAN CITY,2016-W03,19,0,0.0,32.3,23.0,27.65,67.0,8.3,...,NaN,NaN,NaN,NaN,NaN,NaN,1.189623,Low,False,False
2,CALOOCAN CITY,2016-W04,43,0,0.0,30.6,23.8,27.20,65.0,3.9,...,54.55,NaN,70.0,NaN,140.0,NaN,2.691891,Moderate,False,False
3,CALOOCAN CITY,2016-W05,30,0,0.0,32.2,22.6,27.40,67.0,6.4,...,54.85,NaN,66.0,NaN,132.0,NaN,1.877776,Low,False,False
4,CALOOCAN CITY,2016-W06,28,0,0.0,28.3,19.4,23.85,70.0,1.6,...,54.60,109.15,66.0,68.0,132.0,272.0,1.752322,Low,False,False


In [ ]:
df.shape

(4403, 65)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# 1. Identify and handle missing values
print("Missing values before handling:")
print(df.isnull().sum())

# Impute missing numerical values with the mean
numerical_cols = df.select_dtypes(include=np.number).columns
for col in numerical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mean())

# Impute missing categorical values with mode
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after handling all types:")
print(df.isnull().sum())

# 2. Identify categorical features and apply appropriate encoding
categorical_cols_to_encode = []
risk_level_categories = ['Low', 'Moderate', 'High', 'Very High']

if 'RISK_LEVEL' in df.columns:
    df['RISK_LEVEL'] = pd.Categorical(df['RISK_LEVEL'], categories=risk_level_categories, ordered=False)
    categorical_cols_to_encode.append('RISK_LEVEL')
else:
    print("'RISK_LEVEL' column not found. Assuming it has already been one-hot encoded.")

if 'CITY' in df.columns:
    categorical_cols_to_encode.append('CITY')
else:
    print("'CITY' column not found. Assuming it has already been one-hot encoded.")

if categorical_cols_to_encode:
    df = pd.get_dummies(df, columns=categorical_cols_to_encode, drop_first=False)

# Convert boolean columns to integers
for col in ['ALERT', 'EPIDEMIC']:
    if col in df.columns:
        df[col] = df[col].astype(int)
    else:
        print(f"'{col}' column not found. Assuming it has already been converted to int.")

# Convert 'YEAR_WEEK' to numerical format (YYYYww)
def convert_year_week_to_numerical(year_week_str):
    try:
        year_str, week_str = year_week_str.split('-W')
        return int(year_str) * 100 + int(week_str)
    except:
        return np.nan

if 'YEAR_WEEK' in df.columns:
    df['YEAR_WEEK_numerical'] = df['YEAR_WEEK'].apply(convert_year_week_to_numerical)
    df = df.drop('YEAR_WEEK', axis=1, errors='ignore')
    if df['YEAR_WEEK_numerical'].isnull().any():
        df['YEAR_WEEK_numerical'] = df['YEAR_WEEK_numerical'].fillna(df['YEAR_WEEK_numerical'].mean())
else:
    print("'YEAR_WEEK' column not found. Assuming 'YEAR_WEEK_numerical' exists.")

# 🧠 Optional: drop YEAR_WEEK_numerical if not needed
# df = df.drop(columns=['YEAR_WEEK_numerical'], errors='ignore')

# 3. Separate target variable
risk_level_cols = [col for col in df.columns if 'RISK_LEVEL_' in col]
target_cols_existing = [col for col in risk_level_cols if col in df.columns]
y_classification = df[target_cols_existing]

# 4. Drop CASES, DEATHS, and INCIDENCE-related columns (and derived ones)
cols_to_drop_related_to_cases_deaths = [
    col for col in df.columns
    #if any(keyword in col.upper() for keyword in ['CASES', 'DEATHS', 'INCIDENCE'])
    if any(keyword in col.upper() for keyword in ['DEATHS', 'INCIDENCE'])
]

# Drop target + alert/epidemic columns from features
cols_to_drop_from_features = cols_to_drop_related_to_cases_deaths + target_cols_existing + ['CASES', 'ALERT', 'EPIDEMIC']

X_classification = df.drop(columns=cols_to_drop_from_features, errors='ignore')

# 5. Remove any remaining non-numerical columns
non_numerical_cols = X_classification.select_dtypes(exclude=np.number).columns
if len(non_numerical_cols) > 0:
    print(f"\nRemoving non-numerical columns: {list(non_numerical_cols)}")
    X_classification = X_classification.drop(non_numerical_cols, axis=1, errors='ignore')

print("\nColumns in X_classification after dropping CASES/DEATHS/INCIDENCE and non-numerical features:")
print(X_classification.columns)

# 6. Scale numerical features
numerical_cols_classification = X_classification.select_dtypes(include=np.number).columns
if len(numerical_cols_classification) > 0:
    scaler_classification = StandardScaler()
    X_classification[numerical_cols_classification] = scaler_classification.fit_transform(
        X_classification[numerical_cols_classification]
    )

# 7. Train-test split
X_train_classification, X_test_classification, y_train_classification, y_test_classification = train_test_split(
    X_classification, y_classification, test_size=0.2, random_state=42
)

# 8. Reshape for CNN-LSTM
X_train_classification_reshaped = X_train_classification.values.reshape(
    (X_train_classification.shape[0], 1, X_train_classification.shape[1])
)
X_test_classification_reshaped = X_test_classification.values.reshape(
    (X_test_classification.shape[0], 1, X_test_classification.shape[1])
)

print("\n✅ Final Dataset Info:")
print("Shape of training features:", X_train_classification_reshaped.shape)
print("Shape of testing features:", X_test_classification_reshaped.shape)
print("Shape of training target:", y_train_classification.shape)
print("Shape of testing target:", y_test_classification.shape)
print("\nDropped all CASES/DEATHS/INCIDENCE-related columns successfully.")


Missing values before handling:
CITY                   0
YEAR_WEEK              0
CASES                  0
DEATHS                 0
RAINFALL               0
                      ..
RH_roll4_sum          68
INCIDENCE_per_100k     0
RISK_LEVEL             0
ALERT                  0
EPIDEMIC               0
Length: 65, dtype: int64

Missing values after handling all types:
CITY                  0
YEAR_WEEK             0
CASES                 0
DEATHS                0
RAINFALL              0
                     ..
RH_roll4_sum          0
INCIDENCE_per_100k    0
RISK_LEVEL            0
ALERT                 0
EPIDEMIC              0
Length: 65, dtype: int64

Removing non-numerical columns: ['CITY_CALOOCAN CITY', 'CITY_LAS PINAS CITY', 'CITY_MAKATI CITY', 'CITY_MALABON CITY', 'CITY_MANDALUYONG CITY', 'CITY_MANILA CITY', 'CITY_MARIKINA CITY', 'CITY_MUNTINLUPA CITY', 'CITY_NAVOTAS CITY', 'CITY_PARANAQUE CITY', 'CITY_PASAY CITY', 'CITY_PASIG CITY', 'CITY_PATEROS', 'CITY_QUEZON CITY', 'CITY_SA

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Define the CNN-LSTM model for classification
model_classification = Sequential()

# CNN layers - Input shape should match X_train_classification_reshaped
model_classification.add(Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(X_train_classification_reshaped.shape[1], X_train_classification_reshaped.shape[2])))
model_classification.add(MaxPooling1D(pool_size=1))
model_classification.add(Dropout(0.2))

# LSTM layers
model_classification.add(LSTM(50, return_sequences=True))
model_classification.add(Dropout(0.2))
model_classification.add(LSTM(50))
model_classification.add(Dropout(0.2))

# Dense layers
model_classification.add(Dense(50, activation='relu'))
# Output layer for multi-output classification (4 for RISK_LEVEL)
# Use 'sigmoid' activation for multi-label classification
model_classification.add(Dense(y_classification.shape[1], activation='sigmoid'))

# Print the model summary
model_classification.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 1, 64)          │         3,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 1, 50)          │        23,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 50)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           204 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,474 (193.26 KB)

 Trainable params: 49,474 (193.26 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print(X_classification.columns.tolist())

['RAINFALL', 'TMAX', 'TMIN', 'TMEAN', 'RH', 'SUNSHINE', 'POPULATION', 'LAND AREA', 'POP_DENSITY', 'CASES_lag1', 'CASES_lag2', 'CASES_lag3', 'CASES_lag4', 'RAINFALL_lag1', 'RAINFALL_lag2', 'RAINFALL_lag3', 'RAINFALL_lag4', 'TMAX_lag1', 'TMAX_lag2', 'TMAX_lag3', 'TMAX_lag4', 'TMIN_lag1', 'TMIN_lag2', 'TMIN_lag3', 'TMIN_lag4', 'TMEAN_lag1', 'TMEAN_lag2', 'TMEAN_lag3', 'TMEAN_lag4', 'RH_lag1', 'RH_lag2', 'RH_lag3', 'RH_lag4', 'SUNSHINE_lag1', 'SUNSHINE_lag2', 'SUNSHINE_lag3', 'SUNSHINE_lag4', 'CASES_roll2_mean', 'CASES_roll4_mean', 'CASES_roll2_sum', 'CASES_roll4_sum', 'RAINFALL_roll2_mean', 'RAINFALL_roll4_mean', 'RAINFALL_roll2_sum', 'RAINFALL_roll4_sum', 'TMEAN_roll2_mean', 'TMEAN_roll4_mean', 'TMEAN_roll2_sum', 'TMEAN_roll4_sum', 'RH_roll2_mean', 'RH_roll4_mean', 'RH_roll2_sum', 'RH_roll4_sum', 'YEAR_WEEK_numerical']


In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Compile the classification model
# Use binary_crossentropy for multi-label classification with sigmoid activation
model_classification.compile(optimizer=Adam(learning_rate=0.001),
                             loss='binary_crossentropy',
                             metrics=['accuracy'])

print("Classification model compilation complete.")

# Train the classification model
# Using EarlyStopping to prevent overfitting
early_stopping_classification = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_classification = model_classification.fit(X_train_classification_reshaped, y_train_classification,
                                                  epochs=300, batch_size=32, validation_split=0.2,
                                                  callbacks=[early_stopping_classification])

print("Classification model training complete.")

Classification model compilation complete.
Epoch 1/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.4949 - loss: 0.5996 - val_accuracy: 0.5305 - val_loss: 0.4152
Epoch 2/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6101 - loss: 0.3781 - val_accuracy: 0.6284 - val_loss: 0.3652
Epoch 3/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6269 - loss: 0.3524 - val_accuracy: 0.6426 - val_loss: 0.3424
Epoch 4/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6617 - loss: 0.3406 - val_accuracy: 0.6340 - val_loss: 0.3373
Epoch 5/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6835 - loss: 0.3308 - val_accuracy: 0.6738 - val_loss: 0.3233
Epoch 6/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6857 - loss: 0.3198 - val_accuracy: 0.6709 - val_loss: 0.3200
Epoch 7/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6683 - loss: 0.3131 - val_accuracy: 0.6667 - val_loss: 0.3058
Epoch 8/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.

In [ ]:
df.head()

,CASES,DEATHS,RAINFALL,TMAX,TMIN,TMEAN,RH,SUNSHINE,POPULATION,LAND AREA,...,CITY_NAVOTAS CITY,CITY_PARANAQUE CITY,CITY_PASAY CITY,CITY_PASIG CITY,CITY_PATEROS,CITY_QUEZON CITY,CITY_SAN JUAN CITY,CITY_TAGUIG CITY,CITY_VALENZUELA CITY,YEAR_WEEK_numerical
0,27,0,0.0,32.0,21.8,26.90,73.0,6.4,1596900,55.8,...,False,False,False,False,False,False,False,False,False,201602
1,19,0,0.0,32.3,23.0,27.65,67.0,8.3,1597145,55.8,...,False,False,False,False,False,False,False,False,False,201603
2,43,0,0.0,30.6,23.8,27.20,65.0,3.9,1597390,55.8,...,False,False,False,False,False,False,False,False,False,201604
3,30,0,0.0,32.2,22.6,27.40,67.0,6.4,1597635,55.8,...,False,False,False,False,False,False,False,False,False,201605
4,28,0,0.0,28.3,19.4,23.85,70.0,1.6,1597880,55.8,...,False,False,False,False,False,False,False,False,False,201606


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4403 entries, 0 to 4402
Data columns (total 84 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CASES                  4403 non-null   int64  
 1   DEATHS                 4403 non-null   int64  
 2   RAINFALL               4403 non-null   float64
 3   TMAX                   4403 non-null   float64
 4   TMIN                   4403 non-null   float64
 5   TMEAN                  4403 non-null   float64
 6   RH                     4403 non-null   float64
 7   SUNSHINE               4403 non-null   float64
 8   POPULATION             4403 non-null   int64  
 9   LAND AREA              4403 non-null   float64
 10  POP_DENSITY            4403 non-null   int64  
 11  CASES_lag1             4403 non-null   float64
 12  CASES_lag2             4403 non-null   float64
 13  CASES_lag3             4403 non-null   float64
 14  CASES_lag4             4403 non-null   float64
 15  DEAT

In [ ]:
# Train the classification model
# Using EarlyStopping to prevent overfitting
early_stopping_classification = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_classification = model_classification.fit(X_train_classification_reshaped, y_train_classification,
                                                  epochs=300, batch_size=32, validation_split=0.2,
                                                  callbacks=[early_stopping_classification])

print("Classification model training complete.")

Epoch 1/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7439 - loss: 0.2696 - val_accuracy: 0.7106 - val_loss: 0.2874
Epoch 2/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7458 - loss: 0.2726 - val_accuracy: 0.7078 - val_loss: 0.2994
Epoch 3/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7527 - loss: 0.2672 - val_accuracy: 0.7149 - val_loss: 0.2871
Epoch 4/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7460 - loss: 0.2700 - val_accuracy: 0.7121 - val_loss: 0.2834
Epoch 5/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7404 - loss: 0.2602 - val_accuracy: 0.7007 - val_loss: 0.2840
Epoch 6/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7515 - loss: 0.2620 - val_accuracy: 0.7035 - val_loss: 0.2861
Epoch 7/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7481 - loss: 0.2597 - val_accuracy: 0.7106 - val_loss: 0.2887
Epoch 8/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7488 - loss: 0.2627 - val_accuracy: 0.7135 - v

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Evaluate the classification model
loss_classification, accuracy_classification = model_classification.evaluate(X_test_classification_reshaped, y_test_classification, verbose=0)
print(f'Test Loss (Categorical Crossentropy): {loss_classification:.4f}')
print(f'Test Accuracy: {accuracy_classification:.4f}')

# Make predictions on the testing data
predictions_classification = model_classification.predict(X_test_classification_reshaped)

# Convert predicted probabilities to binary labels using a threshold (e.g., 0.5)
predicted_labels = (predictions_classification > 0.5).astype(int)

# The actual labels are already in the correct format
actual_labels = y_test_classification.values

target_names_classification = y_test_classification.columns.tolist()
print("\nClassification Report:")
print(classification_report(actual_labels, predicted_labels, target_names=target_names_classification))


print("\nConfusion Matrix:")
# Confusion matrix for multi-label classification can be computed for each label
# separately or as a single matrix if flattened. Let's compute for each label.
# Reshape actual_labels and predicted_labels to be 1D for confusion matrix
# Or, iterate through each label column to get individual confusion matrices
print("Confusion matrices for each target label:")
for i, target_name in enumerate(target_names_classification):
    print(f"\nConfusion Matrix for {target_name}:")
    print(confusion_matrix(actual_labels[:, i], predicted_labels[:, i]))

Test Loss (Categorical Crossentropy): 0.2855
Test Accuracy: 0.7253
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step

Classification Report:
                      precision    recall  f1-score   support

      RISK_LEVEL_Low       0.87      0.85      0.86       452
 RISK_LEVEL_Moderate       0.59      0.45      0.51       217
     RISK_LEVEL_High       0.59      0.63      0.61       158
RISK_LEVEL_Very High       0.79      0.56      0.65        54

           micro avg       0.75      0.69      0.72       881
           macro avg       0.71      0.62      0.66       881
        weighted avg       0.75      0.69      0.72       881
         samples avg       0.69      0.69      0.69       881


Confusion Matrix:
Confusion matrices for each target label:

Confusion Matrix for RISK_LEVEL_Low:
[[374  55]
 [ 69 383]]

Confusion Matrix for RISK_LEVEL_Moderate:
[[597  67]
 [119  98]]

Confusion Matrix for RISK_LEVEL_High:
[[653  70]
 [ 58 100]]

Confusion Matrix for RISK_LEVEL_Very High:
[[819   8]
 [ 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Display sample predictions
print("\nSample Classification Predictions vs Actuals:")
sample_size = 10
for i in range(sample_size):
    print(f"\nSample {i+1}:")
    predicted_sample = predicted_labels[i]
    actual_sample = actual_labels[i]

    print("Predicted:")
    for j, target_name in enumerate(target_names_classification):
        print(f"  {target_name}: {predicted_sample[j]}")

    print("Actual:")
    for j, target_name in enumerate(target_names_classification):
        print(f"  {target_name}: {actual_sample[j]}")



Sample Classification Predictions vs Actuals:

Sample 1:
Predicted:
  RISK_LEVEL_Low: 1
  RISK_LEVEL_Moderate: 0
  RISK_LEVEL_High: 0
  RISK_LEVEL_Very High: 0
Actual:
  RISK_LEVEL_Low: True
  RISK_LEVEL_Moderate: False
  RISK_LEVEL_High: False
  RISK_LEVEL_Very High: False

Sample 2:
Predicted:
  RISK_LEVEL_Low: 1
  RISK_LEVEL_Moderate: 0
  RISK_LEVEL_High: 0
  RISK_LEVEL_Very High: 0
Actual:
  RISK_LEVEL_Low: True
  RISK_LEVEL_Moderate: False
  RISK_LEVEL_High: False
  RISK_LEVEL_Very High: False

Sample 3:
Predicted:
  RISK_LEVEL_Low: 0
  RISK_LEVEL_Moderate: 1
  RISK_LEVEL_High: 0
  RISK_LEVEL_Very High: 0
Actual:
  RISK_LEVEL_Low: False
  RISK_LEVEL_Moderate: False
  RISK_LEVEL_High: True
  RISK_LEVEL_Very High: False

Sample 4:
Predicted:
  RISK_LEVEL_Low: 1
  RISK_LEVEL_Moderate: 0
  RISK_LEVEL_High: 0
  RISK_LEVEL_Very High: 0
Actual:
  RISK_LEVEL_Low: True
  RISK_LEVEL_Moderate: False
  RISK_LEVEL_High: False
  RISK_LEVEL_Very High: False

Sample 5:
Predicted:
  RISK_LEVEL_Lo

In [ ]:
import os

# Define the path to save the model as a .keras file
model_save_path = "/content/dengue_classification_model.keras"

# Save the classification model in .keras format
model_classification.save(model_save_path)

print(f"Classification model saved successfully to: {model_save_path}")

Classification model saved successfully to: /content/dengue_classification_model.keras


In [ ]:
!pip install joblib

In [ ]:
import joblib
import os

# Define the path to save the scaler
scaler_save_path = "/content/scaler_classification.pkl"

# Save the classification scaler
joblib.dump(scaler_classification, scaler_save_path)

print(f"Classification scaler saved successfully to: {scaler_save_path}")

Classification scaler saved successfully to: /content/scaler_classification.pkl


In [ ]:
# Generate a test case
import numpy as np

# Select a single sample from the test set (e.g., the first sample)
sample_index = 0
test_sample_features = X_test_classification_reshaped[sample_index:sample_index + 1]
test_sample_actual = y_test_classification.iloc[sample_index].values.reshape(1, -1)

# Reshape the scaled test sample features back to 2D for inverse transformation
test_sample_features_2d_scaled = test_sample_features.reshape(1, -1)

# Inverse transform to get the non-scaled features
test_sample_features_2d_non_scaled = scaler_classification.inverse_transform(test_sample_features_2d_scaled)

# Convert back to a DataFrame with original column names for better readability
test_sample_features_non_scaled_df = pd.DataFrame(test_sample_features_2d_non_scaled, columns=X_classification.columns)

# Make a prediction using the trained model
predicted_sample_probabilities = model_classification.predict(test_sample_features)

# Convert predicted probabilities to binary labels using a threshold (e.g., 0.5)
predicted_sample_labels = (predicted_sample_probabilities > 0.5).astype(int)

# Get the target names
target_names_classification = y_test_classification.columns.tolist()

print("Test Case Sample:")
print(f"Input shape (reshaped for model): {test_sample_features.shape}")
print(f"Actual Output shape: {test_sample_actual.shape}")
print(f"Predicted Output shape: {predicted_sample_labels.shape}")

print("\nScaled Feature Values for the test sample:")
print(test_sample_features_2d_scaled)

print("\nNon-Scaled Feature Values for the test sample:")
display(test_sample_features_non_scaled_df)


print("\nPrediction vs Actual for the test sample:")
print("Predicted:")
for j, target_name in enumerate(target_names_classification):
    print(f"  {target_name}: {predicted_sample_labels[0][j]}")

print("Actual:")
for j, target_name in enumerate(target_names_classification):
    print(f"  {target_name}: {test_sample_actual[0][j]}")

# Optional: Compare the prediction to the actual
is_correct = np.array_equal(predicted_sample_labels[0], test_sample_actual[0])
print(f"\nPrediction matches actual: {is_correct}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Test Case Sample:
Input shape (reshaped for model): (1, 1, 54)
Actual Output shape: (1, 4)
Predicted Output shape: (1, 4)

Scaled Feature Values for the test sample:
[[-0.39536533  1.8843769   1.76158374  2.09821455 -1.57356199  1.86710509
  -0.26371402 -0.16709612 -0.33571466 -0.46504251 -0.46667793 -0.44926451
  -0.39364881 -0.39701804 -0.39221879 -0.38765322 -0.38909828  1.55453468
   1.85144323  1.04595005  0.55413215  0.88135498  0.6892402  -0.20108392
  -0.5207294   1.4521402   1.55152476  0.57756765  0.0985788  -1.48112575
  -1.68052403 -1.48557821 -0.78182662  0.98190397  1.77358906  1.58602223
  -0.4928856  -0.47340611 -0.45850735 -0.47340611 -0.45850735 -0.51993034
  -0.6714317  -0.51993034 -0.6714317   1.66947362  1.12470688  1.66947362
   1.12470688 -1.74592411 -1.6379467  -1.74592411 -1.6379467  -1.48366833]]

Non-Scaled Feature Values for the test sample:


,RAINFALL,TMAX,TMIN,TMEAN,RH,SUNSHINE,POPULATION,LAND AREA,POP_DENSITY,CASES_lag1,...,RAINFALL_roll4_sum,TMEAN_roll2_mean,TMEAN_roll4_mean,TMEAN_roll2_sum,TMEAN_roll4_sum,RH_roll2_mean,RH_roll4_mean,RH_roll2_sum,RH_roll4_sum,YEAR_WEEK_numerical
0,0.0,36.3,27.8,32.05,58.0,11.5,592643.0,32.69,18129.0,2.0,...,0.0,31.075,30.15,62.15,120.6,58.0,60.25,116.0,241.0,201617.0



Prediction vs Actual for the test sample:
Predicted:
  RISK_LEVEL_Low: 1
  RISK_LEVEL_Moderate: 0
  RISK_LEVEL_High: 0
  RISK_LEVEL_Very High: 0
Actual:
  RISK_LEVEL_Low: True
  RISK_LEVEL_Moderate: False
  RISK_LEVEL_High: False
  RISK_LEVEL_Very High: False

Prediction matches actual: True
